# FireBridge 1/3: Downloading
Downloads NASA FIRMS active-fire detections and saves them as Parquet on Google Drive.

- **MODIS** (Terra + Aqua, ~1 km pixels): 2000 onward
- **VIIRS** (Suomi NPP, 375 m pixels): 2012 onward

Two ways to download:
- **Study regions** from the team's dataset plan (`NASA_FIRMS_9_Countries_Dataset_Plan.csv`): 7 core and 2 optional regions, 2013–2024. **This is the main one.**
- **Whole countries**, any years.

```
MyDrive/FireBridge/
├── data/raw/         ← 1. Downloading   (sensor=…/country=<area or country>/year=…/data.parquet)
├── data/regions.csv  ← region metadata (bbox, focus period, fire type, goal)
├── data/processed/   ← 2. Preprocessing
└── models/           ← 3. Prediction
```

Raw files keep **every column and every row** inside the region, including gas flares. Cleaning happens in notebook 2.

Runs on a **CPU runtime**. Re-running skips finished files.

In [ ]:
!pip -q install duckdb
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Functions
Run once per session: `downloadRegions`, `downloadCountries`, `listDownloadedCountries`, `readCountries`, `summarizeCountries`, `deleteCountries`, `updateCountries`.

The study regions and their rectangles (bounding boxes) are in the `REGIONS` list at the top. Edit a `bbox` there to change a region.

In [ ]:
import os, time, shutil, datetime, threading
import requests, duckdb, pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# ---------------- configuration ----------------
RAW_DIR = '/content/drive/MyDrive/FireBridge/data/raw'   # Parquet files live here (on Drive)
TMP_DIR = '/content/firebridge_tmp'                     # fast local scratch space, cleaned after each file
FIRMS_BASE = 'https://firms.modaps.eosdis.nasa.gov/data/country'

# Sensors available as FIRMS yearly country files
SENSORS = {
    'modis': {'folder': 'modis',      'first_year': 2000},   # Terra + Aqua, ~1 km pixels
    'viirs': {'folder': 'viirs-snpp', 'first_year': 2012},   # Suomi NPP, 375 m pixels
}

# Column types when converting CSV -> Parquet (everything else is read as text)
_TYPES = {
    'latitude': 'DOUBLE', 'longitude': 'DOUBLE',
    'brightness': 'FLOAT', 'bright_t31': 'FLOAT',      # MODIS
    'bright_ti4': 'FLOAT', 'bright_ti5': 'FLOAT',      # VIIRS
    'scan': 'FLOAT', 'track': 'FLOAT', 'frp': 'FLOAT',
    'acq_date': 'DATE', 'acq_time': 'SMALLINT', 'type': 'TINYINT',
}


# ---------------- helpers (internal) ----------------
# ---------------- helpers (internal) ----------------
def _country_key(name):
    """'United States' -> 'United_States' (the spelling FIRMS uses in file names)."""
    return name.strip().replace(' ', '_')

def _raw_file(sensor, country, year):
    # Hive-style folders: DuckDB reads sensor / country / year straight from the path
    return f'{RAW_DIR}/sensor={sensor}/country={country}/year={year}/data.parquet'

def _csv_to_parquet(csv_path, parquet_path, sensor, bbox=None):
    con = duckdb.connect()
    cols = con.execute(f"SELECT * FROM read_csv('{csv_path}', header=true, all_varchar=true) LIMIT 0").df().columns
    select = []
    for c in cols:
        if c in _TYPES:
            select.append(f'TRY_CAST("{c}" AS {_TYPES[c]}) AS "{c}"')
        elif c == 'confidence' and sensor == 'modis':
            select.append('TRY_CAST(confidence AS SMALLINT) AS confidence')     # MODIS: 0-100
        else:
            select.append(f'"{c}"')                                             # VIIRS confidence stays 'l'/'n'/'h'
    where = ''
    if bbox:                                                                    # keep only the region's rectangle
        lat_min, lat_max, lon_min, lon_max = bbox
        where = (f'WHERE TRY_CAST(latitude AS DOUBLE) BETWEEN {lat_min} AND {lat_max} '
                 f'AND TRY_CAST(longitude AS DOUBLE) BETWEEN {lon_min} AND {lon_max}')
    con.execute(f"""COPY (SELECT {', '.join(select)} FROM read_csv('{csv_path}', header=true, all_varchar=true) {where})
                    TO '{parquet_path}' (FORMAT parquet, COMPRESSION zstd)""")
    con.close()

def _download_one(sensor, country, year, overwrite=False, retries=6, area=None, bbox=None):
    """area/bbox: save only detections inside bbox, under the folder name `area` instead of the country."""
    dest = _raw_file(sensor, area or country, year)
    if os.path.exists(dest) and not overwrite:
        return 'skip'
    folder = SENSORS[sensor]['folder']
    url = f'{FIRMS_BASE}/{folder}/{year}/{folder}_{year}_{country}.csv'
    os.makedirs(TMP_DIR, exist_ok=True)
    tag = f'{sensor}_{area or country}_{year}_{threading.get_ident()}'
    tmp_csv, tmp_pq = f'{TMP_DIR}/{tag}.csv', f'{TMP_DIR}/{tag}.parquet'
    try:
        for attempt in range(retries):                       # FIRMS drops connections now and then
            try:
                with requests.get(url, stream=True, timeout=300) as r:
                    if r.status_code == 404:
                        # FIRMS sometimes answers 404 for files that exist; only trust it after 3 tries.
                        if attempt >= 2:
                            return 'missing'
                        time.sleep(3)
                        continue
                    if r.status_code == 200:
                        with open(tmp_csv, 'wb') as f:      # streamed to local disk, never held in RAM
                            for chunk in r.iter_content(1 << 20):
                                f.write(chunk)
                        break
            except requests.RequestException:
                pass
            time.sleep(2 * (attempt + 1))
        else:
            return 'failed'
        _csv_to_parquet(tmp_csv, tmp_pq, sensor, bbox)      # convert locally (fast) ...
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        shutil.copyfile(tmp_pq, dest + '.tmp')               # ... then one copy to Drive
        os.replace(dest + '.tmp', dest)
        return 'ok'
    except Exception as e:
        print(f'  error {sensor} {area or country} {year}: {e}')
        return 'failed'
    finally:
        for p in (tmp_csv, tmp_pq):
            if os.path.exists(p): os.remove(p)


# ---------------- study regions (from NASA_FIRMS_9_Countries_Dataset_Plan.csv) ----------------
# bbox = (lat_min, lat_max, lon_min, lon_max): a rectangle around each region. It approximates the
# administrative area, so it can include small slices of neighbouring areas.
REGIONS = [
    dict(area='Guinea_Kankan_Siguiri', country='Guinea', priority='Core', bbox=(8.5, 12.5, -10.8, -7.6),
         start='2013-01-01', end='2024-12-31', focus_start='2019-12-01', focus_end='2020-05-31',
         fire_type='Agricultural and seasonal savanna burning', goal='Seasonality'),
    dict(area='Portugal_Centro', country='Portugal', priority='Core', bbox=(39.2, 41.0, -9.0, -6.8),
         start='2013-01-01', end='2024-12-31', focus_start='2017-06-01', focus_end='2017-10-31',
         fire_type='Mediterranean forest wildfire', goal='Anomaly Detection'),
    dict(area='Canada_NWT_South', country='Canada', priority='Core', bbox=(60.0, 64.0, -125.0, -110.0),
         start='2013-01-01', end='2024-12-31', focus_start='2023-05-01', focus_end='2023-09-30',
         fire_type='Boreal forest wildfire', goal='Extreme Fire Activity'),
    dict(area='Australia_NSW', country='Australia', priority='Core', bbox=(-37.6, -28.1, 140.9, 153.7),
         start='2013-01-01', end='2024-12-31', focus_start='2019-10-01', focus_end='2020-02-29',
         fire_type='Bushfires', goal='Southern Hemisphere Seasonality'),
    dict(area='Brazil_North_Pantanal', country='Brazil', priority='Core', bbox=(-18.0, -15.5, -58.2, -55.0),
         start='2013-01-01', end='2024-12-31', focus_start='2020-07-01', focus_end='2020-10-31',
         fire_type='Wetland and vegetation fires', goal='Dry Season Anomaly'),
    dict(area='Indonesia_Central_Kalimantan', country='Indonesia', priority='Core', bbox=(-3.6, 0.9, 110.7, 115.9),
         start='2013-01-01', end='2024-12-31', focus_start='2015-08-01', focus_end='2015-10-31',
         fire_type='Peatland fires', goal='Peatland Fire Anomaly'),
    dict(area='Egypt_Nile_Delta', country='Egypt', priority='Core', bbox=(30.2, 31.65, 30.5, 32.3),
         start='2013-01-01', end='2024-12-31', focus_start='2014-10-01', focus_end='2014-11-30',
         fire_type='Agricultural residue / rice-straw burning', goal='Small-Fire Detection & Seasonality'),
    dict(area='India_Punjab', country='India', priority='Optional', bbox=(29.5, 32.6, 73.8, 77.0),
         start='2013-01-01', end='2024-12-31', focus_start='2020-10-01', focus_end='2020-11-30',
         fire_type='Crop-residue / stubble burning', goal='Small-Fire Sensor Comparison'),
    dict(area='USA_Northern_California', country='United_States', priority='Optional', bbox=(38.0, 42.0, -124.5, -119.9),
         start='2013-01-01', end='2024-12-31', focus_start='2020-08-01', focus_end='2020-10-31',
         fire_type='Forest wildfires', goal='Historical Event Validation'),
]


# ---------------- public functions ----------------
def downloadRegions(areas=None, priority=None, sensors=('modis', 'viirs'), years=None, overwrite=False, workers=6):
    """Download the study regions in REGIONS.

    FIRMS only publishes whole-country files, so each country-year is downloaded in full, then only the
    detections inside the region's rectangle are kept. Large countries (Canada, USA, Brazil, Australia, India)
    still take long to download, but take little space on Drive.

    areas    : list of area names (default: all), e.g. ['Egypt_Nile_Delta', 'Portugal_Centro']
    priority : 'Core' or 'Optional' to pick a group
    years    : override the region's start-end years
    Saves under raw/sensor=.../country=<area>/year=..., so readCountries(['Egypt_Nile_Delta']) works.
    """
    regs = [r for r in REGIONS if (areas is None or r['area'] in areas) and (priority is None or r['priority'] == priority)]
    if not regs:
        print('No matching regions. Names:', [r['area'] for r in REGIONS]); return None
    os.makedirs(RAW_DIR, exist_ok=True)
    pd.DataFrame(REGIONS).to_csv(f'{os.path.dirname(RAW_DIR)}/regions.csv', index=False)   # region metadata for later notebooks
    jobs = []
    for r in regs:
        ys = years if years is not None else range(int(r['start'][:4]), int(r['end'][:4]) + 1)
        for s in sensors:
            jobs += [(s, r['country'], y, r['area'], r['bbox']) for y in ys if y >= SENSORS[s]['first_year']]
    results = []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = {ex.submit(_download_one, s, c, y, overwrite, 6, a, b): (s, a, y) for s, c, y, a, b in jobs}
        for fut in tqdm(as_completed(futures), total=len(futures), desc='Downloading regions'):
            s, a, y = futures[fut]
            results.append({'area': a, 'sensor': s, 'year': y, 'status': fut.result()})
    res = pd.DataFrame(results)
    summary = res.pivot_table(index=['area', 'sensor'], columns='status', values='year', aggfunc='count', fill_value=0).reset_index()
    for col in ('ok', 'skip', 'missing', 'failed'):
        if col not in summary: summary[col] = 0
    if (res.status == 'failed').any():
        print(f"⚠️  {(res.status == 'failed').sum()} file(s) failed. Run downloadRegions again to retry them.")
    return summary[['area', 'sensor', 'ok', 'skip', 'missing', 'failed']]


# ---------------- public functions ----------------
def downloadCountries(countries, sensors=('modis', 'viirs'), years=None, overwrite=False, workers=6):
    """Download all available years of FIRMS fire detections for each country and save them as Parquet.

    countries : list of English country names, e.g. ['Egypt', 'United States']
    sensors   : 'modis' and/or 'viirs'
    years     : optional range/list; default = every year from the sensor's first year to now
    overwrite : re-download files that already exist (default: skip them, so re-running resumes)
    Returns a summary table: one row per country and sensor.
    """
    countries = [_country_key(c) for c in countries]
    this_year = datetime.date.today().year
    jobs = []
    for c in countries:
        for s in sensors:
            ys = years if years is not None else range(SENSORS[s]['first_year'], this_year + 1)
            jobs += [(s, c, y) for y in ys if y >= SENSORS[s]['first_year']]

    results = []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = {ex.submit(_download_one, s, c, y, overwrite): (s, c, y) for s, c, y in jobs}
        for fut in tqdm(as_completed(futures), total=len(futures), desc='Downloading'):
            s, c, y = futures[fut]
            results.append({'country': c, 'sensor': s, 'year': y, 'status': fut.result()})

    res = pd.DataFrame(results)
    summary = (res.pivot_table(index=['country', 'sensor'], columns='status', values='year',
                               aggfunc='count', fill_value=0).reset_index())
    for col in ('ok', 'skip', 'missing', 'failed'):
        if col not in summary: summary[col] = 0
    summary = summary[['country', 'sensor', 'ok', 'skip', 'missing', 'failed']]

    for c in countries:
        r = res[res.country == c]
        if (r.status == 'missing').all():
            print(f"⚠️  '{c}' not found on FIRMS. Check the spelling (English name, e.g. 'United States').")
    failed = res[res.status == 'failed']
    if len(failed):
        print(f'⚠️  {len(failed)} file(s) failed. Run downloadCountries again to retry them '
              f'(finished files are skipped).')
    # The current year is usually 'missing': FIRMS publishes a year's file after the year ends.
    return summary


def listDownloadedCountries():
    """Show what is already downloaded: one row per country and sensor.
    missing_years lists gaps inside the range: run the download again to fill them."""
    rows = []
    if not os.path.isdir(RAW_DIR):
        return pd.DataFrame(columns=['country', 'sensor', 'first_year', 'last_year', 'years', 'size_mb'])
    for sdir in sorted(os.listdir(RAW_DIR)):
        if not sdir.startswith('sensor='): continue
        for cdir in sorted(os.listdir(f'{RAW_DIR}/{sdir}')):
            if not cdir.startswith('country='): continue
            years, size = [], 0
            for ydir in os.listdir(f'{RAW_DIR}/{sdir}/{cdir}'):
                f = f'{RAW_DIR}/{sdir}/{cdir}/{ydir}/data.parquet'
                if ydir.startswith('year=') and os.path.exists(f):
                    years.append(int(ydir[5:])); size += os.path.getsize(f)
            if years:
                gaps = sorted(set(range(min(years), max(years) + 1)) - set(years))
                rows.append({'country': cdir[8:], 'sensor': sdir[7:], 'first_year': min(years),
                             'last_year': max(years), 'years': len(years), 'size_mb': round(size / 1e6, 1),
                             'missing_years': ', '.join(map(str, gaps)) if gaps else ''})
    return pd.DataFrame(rows).sort_values(['country', 'sensor']).reset_index(drop=True)


def readCountries(countries=None, sensors=('modis', 'viirs'), years=None,
                  vegetation_only=True, harmonize=True, output='pandas'):
    """Read downloaded data, loading only what you ask for.

    countries       : list of names (default: all downloaded)
    sensors         : 'modis' and/or 'viirs'
    years           : (start, end) inclusive, e.g. (2012, 2025); default all
    vegetation_only : keep only type == 0 (drops gas flares, industry, offshore)
    harmonize       : same column names for both sensors + numeric confidence + UTC datetime
    output          : 'pandas' -> DataFrame, 'duckdb' -> lazy DuckDB relation (for data too big for RAM)
    """
    parts = []
    for s in sensors:
        glob = f'{RAW_DIR}/sensor={s}/*/*/data.parquet'
        where = []
        if countries:
            names = ', '.join(f"'{_country_key(c)}'" for c in countries)
            where.append(f'country IN ({names})')
        if years:
            where.append(f'year BETWEEN {years[0]} AND {years[1]}')
        if vegetation_only:
            where.append('type = 0')
        w = ('WHERE ' + ' AND '.join(where)) if where else ''
        if harmonize:
            fire, bg = ('brightness', 'bright_t31') if s == 'modis' else ('bright_ti4', 'bright_ti5')
            conf = ('CAST(confidence AS SMALLINT)' if s == 'modis' else
                    "CAST(CASE confidence WHEN 'l' THEN 20 WHEN 'n' THEN 60 WHEN 'h' THEN 90 END AS SMALLINT)")
            conf_cat = ("CASE WHEN confidence < 30 THEN 'low' WHEN confidence < 80 THEN 'nominal' ELSE 'high' END"
                        if s == 'modis' else
                        "CASE confidence WHEN 'l' THEN 'low' WHEN 'n' THEN 'nominal' WHEN 'h' THEN 'high' END")
            sel = f"""latitude, longitude, {fire} AS bright_fire, {bg} AS bright_bg, scan, track, frp,
                      {conf} AS confidence, {conf_cat} AS confidence_cat,
                      acq_date + make_time(acq_time // 100, acq_time % 100, 0) AS datetime,
                      satellite, daynight, type, '{s}' AS sensor, country, CAST(year AS SMALLINT) AS year"""
        else:
            sel = '* REPLACE (CAST(confidence AS VARCHAR) AS confidence)'   # sensor, country, year come from the folder names
        if not any(True for _ in _glob_files(s)):
            continue
        parts.append(f"SELECT {sel} FROM read_parquet('{glob}', hive_partitioning=true) {w}")
    if not parts:
        print('Nothing downloaded yet for these sensors. Run downloadCountries first.')
        return None
    rel = duckdb.sql(' UNION ALL BY NAME '.join(parts))   # default connection, so duckdb.sql() can query the result
    return rel if output == 'duckdb' else rel.df()


def _glob_files(sensor):
    d = f'{RAW_DIR}/sensor={sensor}'
    if os.path.isdir(d):
        for root, _, files in os.walk(d):
            for f in files:
                if f == 'data.parquet':
                    yield f


def summarizeCountries(countries=None, sensors=('modis', 'viirs'), years=None, vegetation_only=True):
    """Fire counts and fire power per country, sensor and year (small table, safe to load)."""
    fires = readCountries(countries, sensors, years, vegetation_only=vegetation_only, output='duckdb')
    if fires is None:
        return None
    return duckdb.sql("""
        SELECT country, sensor, year,
               COUNT(*)                                          AS fires,
               ROUND(SUM(frp))                                   AS total_frp_mw,
               ROUND(AVG(frp), 1)                                AS mean_frp_mw,
               ROUND(100 * AVG(CASE WHEN daynight = 'D' THEN 1 ELSE 0 END)) AS day_pct
        FROM fires
        GROUP BY ALL
        ORDER BY country, sensor, year""").df()


def deleteCountries(countries, sensors=('modis', 'viirs'), confirm=True):
    """Delete downloaded data for these countries (asks for confirmation unless confirm=False)."""
    targets = [f'{RAW_DIR}/sensor={s}/country={_country_key(c)}' for c in countries for s in sensors]
    targets = [t for t in targets if os.path.isdir(t)]
    if not targets:
        print('Nothing to delete.'); return
    print('Will delete:\n  ' + '\n  '.join(t.replace(RAW_DIR + '/', '') for t in targets))
    if confirm and input('Type yes to confirm: ').strip().lower() != 'yes':
        print('Cancelled.'); return
    for t in targets:
        shutil.rmtree(t)
    print(f'Deleted {len(targets)} folder(s).')


def updateCountries(refresh_last_year=False, workers=6):
    """Bring every downloaded country and study region up to date.

    Downloads any newly published years (for example 2026, once FIRMS releases it).
    refresh_last_year=True also re-downloads each one's latest year, in case FIRMS revised it.
    """
    have = listDownloadedCountries()
    if have.empty:
        print('Nothing downloaded yet.'); return None
    areas = {r['area']: r for r in REGIONS}
    if refresh_last_year:
        for _, r in have.iterrows():
            reg = areas.get(r.country)
            _download_one(r.sensor, reg['country'] if reg else r.country, int(r.last_year), overwrite=True,
                          area=r.country if reg else None, bbox=reg['bbox'] if reg else None)
    out = []
    countries = [c for c in have.country.unique() if c not in areas]
    regions = [c for c in have.country.unique() if c in areas]
    if countries:
        out.append(downloadCountries(countries, workers=workers).rename(columns={'country': 'area'}))
    if regions:
        out.append(downloadRegions(regions, years=range(2013, datetime.date.today().year + 1), workers=workers))
    return pd.concat(out, ignore_index=True)

### MODIS 2000–2012 for the historical backfill (needed by notebook 4)
The plan's range starts in 2013. Notebook 4 rebuilds 2000–2012 from MODIS, so download those years too. MODIS only, so this is quick.

In [ ]:
downloadRegions(priority='Core', years=range(2000, 2013), sensors=('modis',))

status,area,sensor,ok,skip,missing,failed
0,Australia_NSW,modis,0,13,0,0
1,Brazil_North_Pantanal,modis,0,13,0,0
2,Canada_NWT_South,modis,0,13,0,0
3,Egypt_Nile_Delta,modis,0,13,0,0
4,Guinea_Kankan_Siguiri,modis,0,13,0,0
5,Indonesia_Central_Kalimantan,modis,0,13,0,0
6,Portugal_Centro,modis,0,13,0,0


## Study regions
FIRMS publishes whole-country files only, so each country-year is downloaded in full and then **cut down to the region's rectangle**. Big countries (Canada, USA, Brazil, Australia, India) take a while to download but end up small on Drive.

Each `bbox` is a rectangle around the region: `(lat_min, lat_max, lon_min, lon_max)`. It approximates the administrative area, so it can include small slices of neighbouring areas.

In [ ]:
pd.DataFrame(REGIONS)[['area', 'country', 'priority', 'bbox', 'start', 'end', 'focus_start', 'focus_end', 'goal']]

,area,country,priority,bbox,start,end,focus_start,focus_end,goal
0,Guinea_Kankan_Siguiri,Guinea,Core,"(8.5, 12.5, -10.8, -7.6)",2013-01-01,2024-12-31,2019-12-01,2020-05-31,Seasonality
1,Portugal_Centro,Portugal,Core,"(39.2, 41.0, -9.0, -6.8)",2013-01-01,2024-12-31,2017-06-01,2017-10-31,Anomaly Detection
2,Canada_NWT_South,Canada,Core,"(60.0, 64.0, -125.0, -110.0)",2013-01-01,2024-12-31,2023-05-01,2023-09-30,Extreme Fire Activity
3,Australia_NSW,Australia,Core,"(-37.6, -28.1, 140.9, 153.7)",2013-01-01,2024-12-31,2019-10-01,2020-02-29,Southern Hemisphere Seasonality
4,Brazil_North_Pantanal,Brazil,Core,"(-18.0, -15.5, -58.2, -55.0)",2013-01-01,2024-12-31,2020-07-01,2020-10-31,Dry Season Anomaly
5,Indonesia_Central_Kalimantan,Indonesia,Core,"(-3.6, 0.9, 110.7, 115.9)",2013-01-01,2024-12-31,2015-08-01,2015-10-31,Peatland Fire Anomaly
6,Egypt_Nile_Delta,Egypt,Core,"(30.2, 31.65, 30.5, 32.3)",2013-01-01,2024-12-31,2014-10-01,2014-11-30,Small-Fire Detection & Seasonality
7,India_Punjab,India,Optional,"(29.5, 32.6, 73.8, 77.0)",2013-01-01,2024-12-31,2020-10-01,2020-11-30,Small-Fire Sensor Comparison
8,USA_Northern_California,United_States,Optional,"(38.0, 42.0, -124.5, -119.9)",2013-01-01,2024-12-31,2020-08-01,2020-10-31,Historical Event Validation


### Download the core regions
Start with the small ones if you want results fast: Egypt, Portugal and Guinea finish quickly. If anything *failed*, run the cell again.

In [ ]:
summary = downloadRegions(priority='Core')
summary

status,area,sensor,ok,skip,missing,failed
0,Australia_NSW,modis,0,12,0,0
1,Australia_NSW,viirs,0,12,0,0
2,Brazil_North_Pantanal,modis,0,12,0,0
3,Brazil_North_Pantanal,viirs,0,12,0,0
4,Canada_NWT_South,modis,0,12,0,0
5,Canada_NWT_South,viirs,0,12,0,0
6,Egypt_Nile_Delta,modis,0,12,0,0
7,Egypt_Nile_Delta,viirs,0,12,0,0
8,Guinea_Kankan_Siguiri,modis,0,12,0,0
9,Guinea_Kankan_Siguiri,viirs,0,12,0,0


### Optional regions (India, USA)

In [ ]:
# downloadRegions(priority='Optional')

### Or just a few


In [ ]:
# downloadRegions(['Egypt_Nile_Delta', 'Portugal_Centro', 'Guinea_Kankan_Siguiri'])

## What's downloaded?
`missing_years` lists gaps inside a range. FIRMS sometimes refuses a file that exists, so if you see gaps, run the download cell again: it only fetches what's missing.

In [ ]:
listDownloadedCountries()

,country,sensor,first_year,last_year,years,size_mb,missing_years
0,Australia,modis,2000,2025,26,86.4,
1,Australia,viirs,2012,2025,14,272.6,
2,Australia_NSW,modis,2000,2024,25,7.7,
3,Australia_NSW,viirs,2013,2024,12,23.0,
4,Brazil,modis,2000,2025,26,135.8,
5,Brazil,viirs,2012,2025,14,325.9,
6,Brazil_North_Pantanal,modis,2000,2024,25,3.2,
7,Brazil_North_Pantanal,viirs,2013,2024,12,7.2,
8,Canada_NWT_South,modis,2000,2024,25,2.8,
9,Canada_NWT_South,viirs,2013,2024,12,16.1,


## Quick check: detections per year
The jump in 2012 is VIIRS starting, not more fires.

In [ ]:
summarizeCountries(['Egypt_Nile_Delta']).pivot_table(index='year', columns='sensor', values='fires')

sensor,modis,viirs
year,,
2000,15.0,NaN
2001,79.0,NaN
2002,230.0,NaN
2003,210.0,NaN
2004,350.0,NaN
2005,480.0,NaN
2006,678.0,NaN
2007,632.0,NaN
2008,559.0,NaN


## Whole countries (optional)
The same data for an entire country, any years (e.g. MODIS back to 2000 for the historical backfill).

In [ ]:
# downloadCountries(['Egypt'])

## Housekeeping

In [ ]:
# Fetch newly published years for every downloaded country and region (e.g. 2026 once FIRMS releases it)
# updateCountries()

In [ ]:
# Remove something you no longer need (asks you to type 'yes')
# deleteCountries(['Egypt_Nile_Delta'])